In [ ]:
# Dataset of Winners Chapel Locations in Nigeria

In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gradio as gr
import os
import csv
import io

In [ ]:
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
openai_api_key = os.getenv('OPENAI_API_KEY')

if hf_token:
    print(f"Huggingface token exists and begins {hf_token[:3]}")
else:
    print("Huggingface token not set")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

login(hf_token, add_to_git_credential=True)
openai = OpenAI()

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
    

In [ ]:
system_message = """
You are generating synthetic location data for Winners Chapel Churches in Nigeria.


Rules:
- Do NOT wrap the output in code blocks.
- The first line must be the header.
- Generate 30 rows of realistic location data.
- Out address must be in Nigeria.
- Include logitude and latitude.
- Do not include empty rows.
"""

user_prompt = f"""
I need you to generate a dataset for Winners Chapel Churches in Nigeria.
"""

messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
]


In [ ]:
def to_csv(csv_text):
    reader = csv.reader(io.StringIO(csv_text))

    with open("output.csv", "w", newline="") as f:
        writer = csv.writer(f)
        for row in reader:
            writer.writerow(row)

In [ ]:
# Wrapping everything in a function - and adding Streaming and generation prompts

def generate_with_oss(max_new_tokens=800):

  model_name = "meta-llama/Llama-3.2-1B-Instruct"

  tokenizer = AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(DEVICE)
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=DEVICE)
  streamer = TextStreamer(tokenizer)
  
  model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
  
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

  # remove prompt tokens
  response = tokenizer.decode(
      outputs[0][input_ids.shape[-1]:],
      skip_special_tokens=True
  )

  to_csv(response)

  return response

In [ ]:
def generate_with_frontier():

  messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

  response = openai.chat.completions.create(model='gpt-4.1-mini', messages=messages)

  to_csv(response.choices[0].message.content)
  
  return response.choices[0].message.content

In [ ]:
with gr.Blocks() as view:
    
    output_box = gr.Textbox(
        label="Output",
        lines=15
    )

    generate_oss_btn = gr.Button("Open Source Model")
    generate_oss_btn.click(
        fn=generate_with_oss,
        inputs=[],
        outputs=output_box
    )

    generate_frontier_btn = gr.Button("Frontier Model")
    generate_frontier_btn.click(
        fn=generate_with_frontier,
        inputs=[],
        outputs=output_box
    )

view.launch(inbrowser=True)